In [ ]:
import pandas as pd
import random
from pprint import pp

from evo.data_converters.common.objects.downhole_collection import DownholeCollection, ColumnMapping, HoleCollars


input_collars_df = pd.DataFrame(
    {
        "hole_index": [1, 2, 3, 4, 5],
        "hole_id": ["BH001", "BH002", "BH003", "BH004", "BH005"],
        "x": [174.7600, 174.7700, 174.7800, 174.7650, 174.7750],
        "y": [-36.8500, -36.8600, -36.8700, -36.8550, -36.8650],
        "z": [10.5, 12.3, 9.8, 11.2, 10.8],
        "final_depth": [25.0, 30.0, 20.0, 28.0, 22.0],
        "date": ["2024-01-15", "2024-01-16", "2024-01-17", "2024-01-18", "2024-01-19"],
        # Additional LOCA attributes
        "LOCA_TYPE": ["BH", "BH", "BH", "BH", "BH"],  # Borehole type
        "LOCA_ENDD": ["2024-01-15", "2024-01-17", "2024-01-18", "2024-01-19", "2024-01-20"],
        "LOCA_LOCM": ["GPS", "GPS", "GPS", "GPS", "GPS"],  # Location method
        "LOCA_LOCX": [0.5, 0.5, 0.5, 0.5, 0.5],  # Location accuracy (m)
        "LOCA_DATM": ["WGS84", "WGS84", "WGS84", "WGS84", "WGS84"],  # Datum
        "LOCA_GREF": ["MSL", "MSL", "MSL", "MSL", "MSL"],  # Ground level reference
        # Additional SCPG attributes
        "SCPG_REF": ["S15.CFIP.A27", "S16.CFIP.A27", "S15.CFIP.A27", "S14.CFIP.A27", "S15.CFIP.A28"],  # Cone reference
        "SCPG_RATE": [20, 15, 22, 18, 20],  # Nominal rate of penetration of the cone
        "SCPG_FRIC": [True, True, True, False, False],  # Friction reducer used
        "SCPG_ENV": [
            "Sunny",
            "Cloudy",
            "Rainy",
            "Overcast",
            "Hurricane",
        ],  # Details of weather and environmental conditions during test
        "SCPG_CONT": ["Alice", "Bob", "Edith", "Alice", "Sally"],  # Subcontractors name
    }
)

input_scpt_df = pd.DataFrame(
    {
        "hole_index": [1, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 4, 4, 4, 4, 5, 5, 5],
        "hole_id": ["BH001"] * 5 + ["BH002"] * 4 + ["BH003"] * 3 + ["BH004"] * 4 + ["BH005"] * 3,
        "SCPT_TESN": ["1"] * 5 + ["1"] * 4 + ["1"] * 3 + ["1"] * 4 + ["1"] * 3,
        "SCPT_DPTH": [0.5, 1.0, 1.5, 2.0, 2.5, 0.5, 1.0, 1.5, 2.0, 0.5, 1.0, 1.5, 0.5, 1.0, 1.5, 2.0, 0.5, 1.0, 1.5],
        "SCPT_RES": [2.5, 3.2, 4.1, 5.2, 6.1, 1.8, 2.9, 3.5, 4.2, 3.1, 4.0, 4.8, 2.2, 3.3, 4.5, 5.5, 2.8, 3.7, 4.3],
        "SCPT_FRES": [1.2, 1.5, 1.3, 1.6, 1.8, 1.1, 1.4, 1.5, 1.7, 1.3, 1.6, 1.4, 1.2, 1.5, 1.7, 1.9, 1.3, 1.5, 1.6],
        "SCPT_PWP1": [
            50,
            100,
            150,
            200,
            250,
            50,
            100,
            150,
            200,
            50,
            100,
            150,
            50,
            100,
            150,
            200,
            50,
            100,
            150,
        ],  # Pore pressure u1
        "SCPT_PWP2": [
            45,
            95,
            145,
            195,
            245,
            45,
            95,
            145,
            195,
            45,
            95,
            145,
            45,
            95,
            145,
            195,
            45,
            95,
            145,
        ],  # Pore pressure u2
        "SCPT_FRR": [
            0.48,
            0.47,
            0.32,
            0.31,
            0.30,
            0.61,
            0.48,
            0.43,
            0.40,
            0.42,
            0.40,
            0.29,
            0.55,
            0.45,
            0.38,
            0.35,
            0.46,
            0.41,
            0.37,
        ],  # Friction ratio
    }
)

# GEOL (Geology) - interval data
lithologies = ["SAND", "CLAY", "SILT", "GRAVEL", "FILL", "SANDSTONE", "SILTSTONE"]
geol_descriptions = [
    "Brown fine to medium SAND",
    "Grey soft CLAY with occasional shell fragments",
    "Brown firm sandy SILT",
    "Grey fine to coarse GRAVEL with sand",
    "Brown loose FILL - sand and gravel",
    "Light grey medium dense to dense SANDSTONE",
    "Grey firm to stiff SILTSTONE",
]

geol_data = {
    "hole_index": [],
    "hole_id": [],
    "GEOL_TOP": [],
    "GEOL_BASE": [],
    "GEOL_DESC": [],
    "GEOL_GEOL": [],
    "GEOL_LEG": [],
    "GEOL_COLR": [],
    "GEOL_DIAM": [],  # Drilling diameter
}

hole_ids = ["BH001", "BH002", "BH003", "BH004", "BH005"]
final_depths = [25.0, 30.0, 20.0, 28.0, 22.0]
colors = ["BROWN", "GREY", "LIGHT GREY", "DARK GREY", "YELLOW BROWN"]

for hole_idx, (hole_id, final_depth) in enumerate(zip(hole_ids, final_depths), start=1):
    cur_depth = 0.0

    while cur_depth < final_depth:
        interval_size = random.uniform(1.5, 4.5)

        if cur_depth + interval_size > final_depth:
            interval_size = final_depth - cur_depth

        lithology_idx = random.randint(0, len(lithologies) - 1)

        geol_data["hole_index"].append(hole_idx)
        geol_data["hole_id"].append(hole_id)
        geol_data["GEOL_TOP"].append(round(cur_depth, 2))
        geol_data["GEOL_BASE"].append(round(cur_depth + interval_size, 2))
        geol_data["GEOL_DESC"].append(geol_descriptions[lithology_idx])
        geol_data["GEOL_GEOL"].append(lithologies[lithology_idx])
        geol_data["GEOL_LEG"].append(f"LEG{lithology_idx + 1}")
        geol_data["GEOL_COLR"].append(random.choice(colors))
        geol_data["GEOL_DIAM"].append(150)  # 150mm drilling diameter

        cur_depth += interval_size

input_geol_df = pd.DataFrame(geol_data)
input_geol_df["GEOL_GEOL"] = input_geol_df["GEOL_GEOL"].astype("category")

collars = HoleCollars(df=input_collars_df)
dhc = DownholeCollection(
    collars=collars,
    measurements=[input_scpt_df, input_geol_df],
    column_mapping=ColumnMapping(DEPTH_COLUMNS=["SCPT_DPTH"], FROM_COLUMNS=["GEOL_TOP"], TO_COLUMNS=["GEOL_BASE"]),
    name="test",
)

for mt in dhc.get_measurement_tables():
    pp(mt.df.columns)

In [ ]:
from python_ags4 import AGS4
from evo.data_converters.common.objects.downhole_collection.tables import DistanceTable

collars_df = dhc.collars.df

# get the first depth table... we'll probably want to loop over all tables and look for ones with SCPT column names.
measurements_df = dhc.get_measurement_tables(filter=[DistanceTable])[0].df

# Create LOCA table from HoleCollars
loca_table = pd.DataFrame(
    {
        "LOCA_ID": collars_df["hole_id"],
        "LOCA_NATE": collars_df["x"],
        "LOCA_NATN": collars_df["y"],
        "LOCA_GL": collars_df["z"],
        "LOCA_STAR": collars_df["date"],
    }
)

# SCPT table from measurements
scpt_table = pd.DataFrame(
    {
        "LOCA_ID": measurements_df["hole_id"],
        "SCPT_DPTH": measurements_df["SCPT_DPTH"],
        "SCPT_RES": measurements_df["SCPT_RES"],
        "SCPT_FRES": measurements_df["SCPT_FRES"],
    }
)

# Create the tables and headings dictionaries
tables = {"LOCA": loca_table, "SCPT": scpt_table}

headings = {"LOCA": loca_table.columns.tolist(), "SCPT": scpt_table.columns.tolist()}

# Write to AGS file
AGS4.dataframe_to_AGS4(tables, headings, "output.ags")